In [1]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over. This repository uses the flat layout
#: `<repo>/notebooks` -> `<repo>`, so the depth is 0.
LOCAL_ROOT_DEPTH = 0


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


# Verification — `CrowdSegmenter_CT_GCE` against `research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r09.md`

Executed report, not the source of truth. Every mathematical claim lives as a
test under `tests/`; this page runs them and shows the evidence. Existence of
this file proves nothing — `verify` reads whether its cells actually ran.

Re-run headlessly from the repository root:

```
PATH="$PWD/.venv/bin:$PATH" .venv/bin/python -m jupyter nbconvert \
  --to notebook --execute --inplace \
  notebooks/verification.ipynb
```

The `PATH` prefix is not decoration. The kernelspec a notebook records launches
a bare `python` with no absolute path, so the kernel runs whatever is first on
`PATH` — not the interpreter that launched jupyter. Drop the prefix and the
command reads as isolated and is not: the report gets written by an interpreter
that is not this project's, and nothing in the output says it.


In [2]:
import ast
import sys

import torch

sys.path.insert(0, str(ROOT / "src"))
rng = torch.Generator().manual_seed(7)

# Which revision each module was written against, read statically.
for file in sorted((ROOT / "src" / "CrowdSegmenter_CT_GCE").rglob("*.py")):
    if file.name == "__init__.py":
        continue
    for node in ast.parse(file.read_text()).body:
        if any(getattr(t, "id", None) == "__provenance__" for t in getattr(node, "targets", [])):
            p = ast.literal_eval(node.value)
            print(f"{file.name:<18} {p['revision']:<26} eqs {','.join(p['equations'])}")
# The fingerprint of the code this report ran against. `execution_count` proves a
# cell ran once; it says nothing about what it ran against, and a report executed
# once and never again stays green while the code moves out from under it. `verify`
# recomputes this digest and compares — so the notebook must not compute one of its
# own: a second formula agrees with the checker only by coincidence, and this one
# did not. `report_digest` is the single implementation, scaffolded beside the
# method's own package, and `src/` is already on the path above.
from CrowdSegmenter_CT_GCE import report_digest

print(report_digest.stamp())


bound_study.py     research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md eqs 1,2,3
dawid_skene.py     research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md eqs 3
design_round.py    research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md eqs 1
identifiability.py research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md eqs 5,6,7
metric.py          research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md eqs 8,9
metric_pilot.py    research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md eqs 8,9
module.py          research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md eqs 1,2
reliability.py     research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md eqs 1,2
selection_study.py research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md 

SOURCES-SHA256 c609eb535fa3deb66bc39f4305a1a14982a348013a3bb2c53d9cd0735f08de43


## Levels 1, 2, 4 and 5 — the suite

A red cell here means the implementation does not match what the proposal
claims. Do not edit the assertion to make it pass.

In [3]:
import pytest

code = pytest.main(["-q", str(ROOT / "tests"), "--rootdir", str(ROOT)])
assert code == 0, f"test suite failed (pytest exit code {code})"

.

.

.

.

.

.

... (70 outputs left out)

.

.

                                                                   [100%]


=============================== warnings summary ===============================
tests/test_metric_pilot.py::test_pilot_verdicts_require_floor_and_separation
  /home/lucas/papers/ct-gce-smith/implementations/CrowdSegmenter-CT-GCE/src/CrowdSegmenter_CT_GCE/metric_pilot.py:280: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
    rho, pval = spearmanr(means, np.asarray(supervised, dtype=np.float64))

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
77 passed, 1 skipped, 1 warning in 9.26s


## Level 3 — synthetic evidence

Deterministic, fixed seed, ground truth known by construction. State the
expected behaviour before looking at the output.

In [4]:
# Synthetic evidence, deterministic. The suite under tests/ is the source of
# truth; this cell shows the objects on one hand-built case.
import torch

from CrowdSegmenter_CT_GCE.module import ctgce_grad_lambda, ctgce_loss
from CrowdSegmenter_CT_GCE.reliability import unconstrained_optimum
from CrowdSegmenter_CT_GCE.thresholds import calibrate_cutoffs

torch.set_printoptions(precision=4, sci_mode=False)

# Eq. (1)-(2): R = 2 annotators, three pixels, K = 3 categories.
P = torch.tensor([[0.70, 0.20, 0.10],
                  [0.10, 0.80, 0.10],
                  [0.20, 0.30, 0.50]], dtype=torch.float64)
Y = torch.zeros(2, 3, 3, dtype=torch.float64)
for r, row in enumerate([[0, 1, 2], [0, 2, 1]]):
    for s, k in enumerate(row):
        Y[r, s, k] = 1.0
M = torch.ones(2, 3, dtype=torch.float64)
Lambda = torch.full((2, 3, 3), 0.5, dtype=torch.float64)
C = torch.tensor([0.40, 0.30, 0.20], dtype=torch.float64)
q = 0.7

print("Eq. (1) CT-GCE loss:", float(ctgce_loss(P, Y, M, Lambda, C, q)))
print("Eq. (2) d/dLambda at the annotated class:")
print(ctgce_grad_lambda(P, Y, M, C, q))
print("Eq. (2) unconstrained optimum (the gate):")
print(unconstrained_optimum(P, Y, M, C, q))

# Eq. (4): one category whose 1/K floor binds. The honest discarded fraction
# is F_k(C_k), not rho_hat_k; tests/findings.py measures the gap over 200
# configurations.
K = 3
P_thr = torch.zeros(9, K, dtype=torch.float64)
Y_thr = torch.zeros(1, 9, K, dtype=torch.float64)
M_thr = torch.ones(1, 9, dtype=torch.float64)
labels = [0] * 6 + [1] * 2 + [2]
values = [0.05, 0.10, 0.15, 0.20, 0.90, 0.95, 0.20, 0.30, 0.40]
for i, (k, v) in enumerate(zip(labels, values)):
    P_thr[i, k] = v
    rest = (1.0 - v) / (K - 1)
    for j in range(K):
        if j != k:
            P_thr[i, j] = rest
    Y_thr[0, i, k] = 1.0

rho_hat = torch.tensor([0.10, 0.10, 0.10], dtype=torch.float64)
calibration = calibrate_cutoffs(P_thr, Y_thr, M_thr, rho_hat)
print("Eq. (4) cutoffs:", calibration.cutoffs)
print("Eq. (4) clipped at 1/K:", calibration.clipped)
print("rho_hat:", rho_hat)
print("discarded fraction F_k(C_k):", calibration.discarded)


Eq. (1) CT-GCE loss: 0.6879165230479177
Eq. (2) d/dLambda at the annotated class:
tensor([[[-0.3607,  0.0000,  0.0000],
         [ 0.0000, -0.6070,  0.0000],
         [ 0.0000,  0.0000, -0.4163]],

        [[-0.3607,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.1780],
         [ 0.0000,  0.0000,  0.0000]]], dtype=torch.float64)
Eq. (2) unconstrained optimum (the gate):
tensor([[[1., 0., 0.],
         [0., 1., 0.],
         [0., 0., 1.]],

        [[1., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]], dtype=torch.float64)
Eq. (4) cutoffs: tensor([0.3333, 0.3333, 0.4000], dtype=torch.float64)
Eq. (4) clipped at 1/K: tensor([ True,  True, False])
rho_hat: tensor([0.1000, 0.1000, 0.1000], dtype=torch.float64)
discarded fraction F_k(C_k): tensor([0.6667, 1.0000, 1.0000], dtype=torch.float64)
